# Task 4 - Graph topology ingestion into Neo4j

Kafka Connect consumes only node and edge topics. Cypher handlers use `MERGE`, create placeholder endpoints when necessary, and reconcile delete events without Spark.

In [1]:
!docker compose exec -T connect curl -fsS http://localhost:8083/connectors/cpg-neo4j-sink/status

{"name":"cpg-neo4j-sink","connector":{"state":"RUNNING","worker_id":"connect:8083"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"connect:8083"}],"type":"sink"}


In [1]:
import subprocess
queries = ['MATCH (n:CPGNode) WITH count(n) AS nodes, count(DISTINCT n.id) AS unique_nodes MATCH ()-[r:CPG_EDGE]->() RETURN nodes, unique_nodes, count(r) AS edges, count(DISTINCT r.id) AS unique_edges', 'MATCH ()-[r:CPG_EDGE]->() RETURN r.kind AS kind, count(*) AS count ORDER BY kind']
password = subprocess.run(
    ['docker', 'compose', 'exec', '-T', 'neo4j', 'printenv', 'LAB04_NEO4J_PASSWORD'],
    capture_output=True, text=True, check=True,
).stdout.strip()
for query in queries:
    result = subprocess.run(
        ['docker', 'compose', 'exec', '-T', 'neo4j', 'cypher-shell', '-u', 'neo4j', '-p', password],
        input=query, capture_output=True, text=True, check=True,
    )
    print(result.stdout.rstrip())

nodes, unique_nodes, edges, unique_edges
62397, 62397, 77849, 77849
kind, count
"AST", 57960
"CALL", 2593
"CFG", 7248
"DFG", 10048


In [1]:
!docker compose exec -T broker kafka-get-offsets --bootstrap-server broker:29092 --topic cpg.neo4j-dlq.v1

cpg.neo4j-dlq.v1:0:0


![Neo4j Browser showing CPG nodes and relationships](figures/neo4j-browser.png)

*Neo4j Browser showing CPG nodes and relationships*

## Reflection

Total and distinct counts are identical, so forced replay did not duplicate topology. The connector task is RUNNING and its DLQ is checked separately.